### 35 clips part for testing if our N is make sense and doable

In [ ]:
import pandas as pd

final_sample = pd.read_csv("final_sample_600_per_group.csv")
timing_test = (
    final_sample
    .groupby("primary_accent", group_keys=False)
    .apply(lambda g: g.sample(n=5, random_state=42))
)
timing_test.to_csv("timing_test_35_clips.csv", index=False)
print(timing_test.groupby("primary_accent").size())
timing_test

In [ ]:
import tarfile
from pathlib import Path

TAR_PATH = Path("D:/DS/COMPS760_Project/1781724951333-cv-corpus-26.0-2026-06-12-en.tar.gz")
needed = set(timing_test["path"])
print(f"need {len(needed)} files")

OUTPUT_DIR = Path("timing_test_audio")
OUTPUT_DIR.mkdir(exist_ok=True)

found = 0
with tarfile.open(TAR_PATH, mode="r|gz") as tar:
    for member in tar:
        if Path(member.name).name in needed:
            tar.extract(member, path=OUTPUT_DIR)
            found += 1
            print(f"  [{found}/{len(needed)}] {Path(member.name).name}")
            if found == len(needed):
                break

print(f"\nFinished: {found} / {len(needed)}")

### Full clips part

In [ ]:
import pandas as pd
import tarfile
from pathlib import Path
import psutil

In [ ]:
PROJECT_DIR = Path("D:/DS/COMPS760_Project")

final_sample = pd.read_csv(PROJECT_DIR / "final_sample_600_per_group.csv")
needed_df = final_sample[["path", "primary_accent"]].drop_duplicates("path")
needed = set(needed_df["path"])
print(f"need {len(needed)} files")
print(needed_df["primary_accent"].value_counts())

In [ ]:
psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS) #just for lower priority, so I can still using my pc while it running

TAR_PATH = PROJECT_DIR / "1781724951333-cv-corpus-26.0-2026-06-12-en.tar.gz"
OUTPUT_DIR = PROJECT_DIR / "final_sample_audio"
OUTPUT_DIR.mkdir(exist_ok=True)

found_paths = set()
scanned = 0
with tarfile.open(TAR_PATH, mode="r|gz") as tar:
    for member in tar:
        scanned += 1
        name = Path(member.name).name
        if name in needed and name not in found_paths:
            tar.extract(member, path=OUTPUT_DIR)
            found_paths.add(name)

        if scanned % 100_000 == 0:
            print(f"  scanned {scanned:,}, found {len(found_paths)} / {len(needed)}")

        if len(found_paths) == len(needed):
            break

print(f"\nfinished: {len(found_paths)} / {len(needed)}, scanned {scanned:,} total")

In [ ]:
missing = needed - found_paths
if missing:
    missing_df = needed_df[needed_df["path"].isin(missing)]
    print(f"missing {len(missing)}:")
    print(missing_df["primary_accent"].value_counts())
else:
    print("no missing files")

In [ ]:
files = list(OUTPUT_DIR.glob("*.mp3"))
print(f"file number: {len(files)}")
sizes = [f.stat().st_size for f in files]
print(f"Total size: {sum(sizes)/1e6:.1f} MB")
print(f"Smallest: {min(sizes)} bytes, Biggest: {max(sizes)} bytes")

zero_byte = [f for f in files if f.stat().st_size == 0]
if zero_byte:
    print(f"⚠️ {len(zero_byte)} file is empty，need to re sub-sampling：{[f.name for f in zero_byte[:5]]}")
else:
    print("No 0 byte file")